In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r"C:\Users\HP\Downloads\Dataset .csv")
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [4]:
features = ['Cuisines', 'Price range', 'Aggregate rating']
df = df[features]

In [5]:
df.isnull().sum()

Cuisines            9
Price range         0
Aggregate rating    0
dtype: int64

In [6]:
df.fillna("Unknown", inplace=True)

In [8]:
df['combined_features'] = df['Cuisines'].astype(str) + " " + \
                          df['Price range'].astype(str) + " " + \
                          df['Aggregate rating'].astype(str)

In [9]:
df.head()

,Cuisines,Price range,Aggregate rating,combined_features
0,"French, Japanese, Desserts",3,4.8,"French, Japanese, Desserts 3 4.8"
1,Japanese,3,4.5,Japanese 3 4.5
2,"Seafood, Asian, Filipino, Indian",4,4.4,"Seafood, Asian, Filipino, Indian 4 4.4"
3,"Japanese, Sushi",4,4.9,"Japanese, Sushi 4 4.9"
4,"Japanese, Korean",4,4.8,"Japanese, Korean 4 4.8"


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
feature_matrix = tfidf.fit_transform(df['combined_features'])

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

In [15]:
cuisine = input("Enter preferred cuisine: ")
price = input("Enter price range (1-4): ")
rating = input("Enter minimum rating (e.g., 4.0): ")

def create_preference(cuisine, price, rating):
    return f"{cuisine} {price} {rating}"

user_pref = create_preference(cuisine, price, rating)

def recommend_restaurants(user_pref, df, tfidf, feature_matrix, top_n=5):
    user_vector = tfidf.transform([user_pref])
    
    similarity = cosine_similarity(user_vector, feature_matrix)
    
    similar_indices = similarity.argsort()[0][-top_n:][::-1]
    
    return df.iloc[similar_indices]
    
user_pref = create_preference(cuisine, price, rating)

recommendations = recommend_restaurants(
    user_pref, df, tfidf, feature_matrix
)

print(recommendations)

Enter preferred cuisine:  Chinese
Enter price range (1-4):  1
Enter minimum rating (e.g., 4.0):  3.0


     Cuisines  Price range  Aggregate rating combined_features
2002  Chinese            1               0.0     Chinese 1 0.0
7459  Chinese            1               0.0     Chinese 1 0.0
7600  Chinese            2               0.0     Chinese 2 0.0
7593  Chinese            1               0.0     Chinese 1 0.0
7590  Chinese            1               0.0     Chinese 1 0.0
